# WM-01 · DIAMOND 世界模型（Atari 预训练）

**目标**：在 Kaggle T4 上加载 [DIAMOND](https://github.com/eloialonso/diamond) 预训练扩散世界模型，做 **headless 想象 rollout**，导出 GIF。

**说明**：交互 `play.py` 需要 GUI；本 notebook 用 API 在「梦境」里滚帧。不碰 CS:GO 大模型。

渐进：环境检查 → 装依赖 → clone → 下权重 → 收集初始化帧 → 世界模型想象 → 导出

In [ ]:
import os, sys, subprocess, traceback
from pathlib import Path
import torch
print(sys.version)
print('cuda', torch.cuda.is_available(), torch.__version__)
assert torch.cuda.is_available(), '请打开 GPU'
print('GPU', torch.cuda.get_device_name(0))
print('VRAM_GB', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))
WORK = Path('/kaggle/working')
OUT = WORK/'wm_diamond'
OUT.mkdir(exist_ok=True)
os.chdir(WORK)

In [ ]:
import subprocess, sys
from pathlib import Path

def pip(*args):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', *args]
    print('>>', ' '.join(cmd[-8:]))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode:
        print((r.stderr or r.stdout or '')[-2000:])
    return r.returncode == 0

# 不强制降级/重装 torch 与 numpy（Kaggle 易 ABI 炸）
pip('gymnasium==0.29.1', 'ale-py==0.9.0', 'h5py', 'huggingface-hub', 'hydra-core==1.3',
    'opencv-python-headless', 'pillow', 'pygame', 'torcheval', 'tqdm', 'omegaconf', 'einops')
# AutoROM 接受协议拉 ROM
pip('autorom[accept-rom-license]')
try:
    subprocess.run([sys.executable, '-m', 'AutoROM', '--accept-license'], check=False)
except Exception as e:
    print('AutoROM', e)
print('deps done')

In [ ]:
import subprocess
from pathlib import Path
WORK = Path('/kaggle/working')
REPO = WORK/'diamond'
if not REPO.exists():
    subprocess.run('git clone --depth 1 https://github.com/eloialonso/diamond.git ' + str(REPO), shell=True, check=True)
print('repo', REPO.exists(), list(REPO.iterdir())[:8])

In [ ]:

import os, sys, traceback
from pathlib import Path
import torch
import numpy as np
from PIL import Image

WORK = Path('/kaggle/working')
REPO = WORK/'diamond'
OUT = WORK/'wm_diamond'
OUT.mkdir(exist_ok=True)
# imageio
import subprocess
subprocess.run([sys.executable,'-m','pip','install','-q','imageio'], check=False)
import imageio.v2 as imageio

os.chdir(REPO)
sys.path.insert(0, str(REPO/'src'))

GAME = 'Breakout'
N_COLLECT = 200
N_IMAGINE = 60
errors = []
ok = False

try:
    from huggingface_hub import hf_hub_download
    from hydra import compose, initialize_config_dir
    from hydra.utils import instantiate
    from omegaconf import OmegaConf
    from torch.utils.data import DataLoader

    OmegaConf.register_new_resolver('eval', eval, replace=True)

    def dl(fn):
        return Path(hf_hub_download('eloialonso/diamond', fn))

    path_ckpt = dl(f'atari_100k/models/{GAME}.pt')
    agent_yaml = dl('atari_100k/config/agent/default.yaml')
    env_yaml = dl('atari_100k/config/env/atari.yaml')

    cfg_dir = str((REPO/'config').resolve())
    with initialize_config_dir(version_base='1.3', config_dir=cfg_dir):
        cfg = compose(config_name='trainer')

    # 挂到完整 cfg 树上再 instantiate，才能解析 ${agent...} / ${env...}
    cfg.agent = OmegaConf.load(agent_yaml)
    cfg.env = OmegaConf.load(env_yaml)
    cfg.env.train.id = cfg.env.test.id = f'{GAME}NoFrameskip-v4'
    cfg.world_model_env.horizon = N_IMAGINE
    # 关闭可能有问题的 compile
    try:
        cfg.training.compile_wm = False
    except Exception:
        pass

    from agent import Agent
    from coroutines.collector import make_collector, NumToCollect
    from data import BatchSampler, collate_segments_to_batch, Dataset
    # PyTorch 2.x: Sampler.__init__ 不再接收 dataset，DIAMOND 旧代码会炸
    import torch.utils.data as tud
    _si = tud.Sampler.__init__
    def _sampler_init_compat(self, *args, **kwargs):
        try:
            return _si(self, *args, **kwargs)
        except TypeError:
            return _si(self)
    tud.Sampler.__init__ = _sampler_init_compat
    from envs import make_atari_env, WorldModelEnv

    device = torch.device('cuda:0')
    test_env = make_atari_env(num_envs=1, device=device, **cfg.env.test)
    agent = Agent(instantiate(cfg.agent, num_actions=test_env.num_actions)).to(device).eval()
    agent.load(path_ckpt)
    print('loaded', path_ckpt, 'num_actions', test_env.num_actions)

    ds_path = Path(f'dataset/{GAME}_{N_COLLECT}')
    dataset = Dataset(ds_path)
    dataset.load_from_default_path()
    if len(dataset) == 0:
        print('collecting', N_COLLECT, 'real steps...')
        collector = make_collector(test_env, agent.actor_critic, dataset, epsilon=0)
        collector.send(NumToCollect(steps=N_COLLECT))
        dataset.save_to_default_path()
    print('dataset', len(dataset))

    ncond = cfg.agent.denoiser.inner_model.num_steps_conditioning
    bs = BatchSampler(dataset, 0, 1, 1, ncond, None, False)
    dl_loader = DataLoader(dataset, batch_sampler=bs, collate_fn=collate_segments_to_batch)
    wm_env_cfg = instantiate(cfg.world_model_env, num_batches_to_preload=1)
    wm_env = WorldModelEnv(agent.denoiser, agent.rew_end_model, dl_loader, wm_env_cfg, return_denoising_trajectory=False)

    obs, info = wm_env.reset()
    frames = []

    def to_img(o):
        x = o
        if isinstance(x, (tuple, list)):
            x = x[0]
        if isinstance(x, dict):
            x = x.get('obs', x.get('image', next(iter(x.values()))))
        if torch.is_tensor(x):
            x = x.detach().float().cpu()
            if x.ndim == 4:
                x = x[0]
            if x.ndim == 3 and x.shape[0] in (1, 3, 4):
                x = x[:3].permute(1, 2, 0).numpy()
            else:
                x = x.numpy()
        x = np.asarray(x)
        if x.max() <= 1.5:
            x = x * 255.0
        x = x.clip(0, 255).astype(np.uint8)
        if x.ndim == 2:
            x = np.stack([x, x, x], -1)
        return x

    frames.append(to_img(obs))
    for t in range(N_IMAGINE):
        try:
            a = agent.actor_critic.act(obs, should_sample=True)
        except Exception:
            try:
                a = agent.actor_critic.act(obs)
            except Exception:
                a = torch.randint(0, test_env.num_actions, (1,), device=device)
        step = wm_env.step(a)
        obs = step[0] if isinstance(step, tuple) else step
        frames.append(to_img(obs))
        if (t + 1) % 15 == 0:
            print('imagine', t + 1)

    gif_path = OUT / f'diamond_{GAME}_dream.gif'
    # upscale frames for visibility
    up = []
    for f in frames:
        im = Image.fromarray(f).resize((210, 210), Image.NEAREST)
        up.append(np.array(im))
    imageio.mimsave(gif_path, up, fps=12, loop=0)
    idxs = np.linspace(0, len(up) - 1, num=min(8, len(up)), dtype=int)
    imgs = [Image.fromarray(up[i]) for i in idxs]
    sheet = Image.new('RGB', (210 * len(imgs), 210))
    for i, im in enumerate(imgs):
        sheet.paste(im, (i * 210, 0))
    sheet_path = OUT / f'diamond_{GAME}_strip.png'
    sheet.save(sheet_path)
    print('GIF', gif_path, 'frames', len(frames))
    print('STRIP', sheet_path)
    try:
        from IPython.display import display, Image as IImage
        display(IImage(filename=str(gif_path)))
        display(sheet)
    except Exception:
        pass
    ok = True
except Exception:
    errors.append(traceback.format_exc())
    print(traceback.format_exc())

import shutil
shutil.make_archive(str(WORK / 'wm_diamond_export'), 'zip', OUT)
print('ZIP', WORK / 'wm_diamond_export.zip')
assert ok, 'DIAMOND failed:\n' + '\n'.join(errors)[:2500]
print('01 DIAMOND DONE')
